In [ ]:
# PATHS ABSOLUTOS EXACTOS
PATH_TEAM_GAMELOGS = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00d_featurized/2024-25/teamgamelogs_featurized.parquet"
PATH_BOXSCORES = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/boxscores.parquet"
PATH_ON = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_1.parquet"
PATH_OFF = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_2.parquet"
PATH_LINEUPS = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_dash_lineups__dataset_1.parquet"

In [ ]:
from pathlib import Path
import pandas as pd

def _load_parquet(path_str):
    path = Path(path_str)
    if not path.exists():
        print(f'⚠️ Archivo no encontrado: {path}')
        return None
    df = pd.read_parquet(path)
    print(f'{path.name}: shape={df.shape}')
    print(f'Columnas ({len(df.columns)}): {sorted(df.columns)}')
    return df

df_team = _load_parquet(PATH_TEAM_GAMELOGS)
df_box = _load_parquet(PATH_BOXSCORES)
df_on = _load_parquet(PATH_ON)
df_off = _load_parquet(PATH_OFF)
df_lineups = _load_parquet(PATH_LINEUPS)


In [ ]:
# from 02_FeatureFunctions import add_lineup_features_in_memory, DEFAULT_LINEUP_CONFIG
import importlib

_feature_module = importlib.import_module('02_FeatureFunctions')
add_lineup_features_in_memory = _feature_module.add_lineup_features_in_memory
DEFAULT_LINEUP_CONFIG = _feature_module.DEFAULT_LINEUP_CONFIG


In [ ]:
from copy import deepcopy

config_safe = deepcopy(DEFAULT_LINEUP_CONFIG)
config_safe['USE_ON_OFF'] = False

safe_on = pd.DataFrame()
safe_off = pd.DataFrame()

if df_team is not None and df_box is not None:
    df_augmented = add_lineup_features_in_memory(
        df_teamgames=df_team,
        df_player_box=df_box,
        df_on=safe_on,
        df_off=safe_off,
        df_lineups=df_lineups,
        config=config_safe,
    )
    display(df_augmented.head())
else:
    df_augmented = None
    print('⚠️ No se generaron features LINEUP_* por falta de datos base.')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if df_augmented is not None:
    lineup_cols = [c for c in df_augmented.columns if c.startswith('LINEUP_')]
    if lineup_cols:
        display(df_augmented[lineup_cols].describe(include='all'))
        plt.figure(figsize=(6, 4))
        df_augmented['LINEUP_SCORE'].dropna().plot.hist(bins=20, alpha=0.7)
        plt.title('Distribución LINEUP_SCORE')
        plt.xlabel('LINEUP_SCORE')
        plt.show()
        if 'W' in df_augmented.columns:
            corr_value = df_augmented[['LINEUP_SCORE', 'W']].dropna().corr().iloc[0, 1]
            print(f'Correlación LINEUP_SCORE vs W: {corr_value:.4f}')
        elif 'WL' in df_augmented.columns:
            wl_numeric = df_augmented['WL'].map({'W': 1, 'L': 0})
            corr_value = df_augmented[['LINEUP_SCORE']].assign(WL_NUM=wl_numeric).dropna().corr().iloc[0, 1]
            print(f'Correlación LINEUP_SCORE vs WL: {corr_value:.4f}')
        plt.figure(figsize=(6, 4))
        corr_matrix = df_augmented[lineup_cols].corr()
        sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f')
        plt.title('Correlación entre features LINEUP_*')
        plt.show()
    else:
        print('No se encontraron columnas LINEUP_* para analizar.')
else:
    print('Sin datos enriquecidos para analizar.')


In [ ]:
import pandas as pd

if df_augmented is not None:
    df_sorted = df_augmented.sort_values(['TEAM_ID', 'GAME_DATE'])
    date_diff_check = df_sorted.groupby('TEAM_ID')['GAME_DATE'].diff().dropna()
    has_future_leak = (date_diff_check < pd.Timedelta(0)).any()
    score_bounds = df_augmented['LINEUP_SCORE'].dropna()
    print(f'Fechas en orden cronológico por equipo: {not has_future_leak}')
    if not score_bounds.empty:
        print('LINEUP_SCORE dentro de [0, 1]:', bool(((score_bounds >= 0) & (score_bounds <= 1)).all()))
    starters_non_negative = df_augmented['LINEUP_STARTERS_OUT'].dropna()
    if not starters_non_negative.empty:
        print('LINEUP_STARTERS_OUT sin valores negativos:', bool((starters_non_negative >= 0).all()))
else:
    print('Sin datos para validar leakage temporal.')
